In [1]:
import pandas as pd
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma

load_dotenv()

True

In [2]:
df = pd.read_csv("../data/processed/cleaned_netflix.csv")

df.head()

,Show_Id,Category,Title,Director,Cast,Country,Release_Date,Rating,Duration,Type,Description,combined_text
0,s1,TV Show,3%,Unknown,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,"August 14, 2020",TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...,Title: 3%. Category: TV Show. Genre: Internati...
1,s2,Movie,07:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,"December 23, 2016",TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...,"Title: 07:19. Category: Movie. Genre: Dramas, ..."
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,"December 20, 2018",R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow...",Title: 23:59. Category: Movie. Genre: Horror M...
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,"November 16, 2017",PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi...",Title: 9. Category: Movie. Genre: Action & Adv...
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,"January 1, 2020",PG-13,123 min,Dramas,A brilliant group of students become card-coun...,Title: 21. Category: Movie. Genre: Dramas. Dir...


In [3]:
df["combined_text"].head()

0    Title: 3%. Category: TV Show. Genre: Internati...
1    Title: 07:19. Category: Movie. Genre: Dramas, ...
2    Title: 23:59. Category: Movie. Genre: Horror M...
3    Title: 9. Category: Movie. Genre: Action & Adv...
4    Title: 21. Category: Movie. Genre: Dramas. Dir...
Name: combined_text, dtype: object

In [4]:
df = df.sample(500, random_state=36)

In [5]:
documents = []

for _, row in df.iterrows():
    document = Document(
        page_content=row["combined_text"],
        metadata={
            "title": row["Title"],
            "category": row["Category"],
            "genre": row["Type"]
        }
    )
    
    documents.append(document)

print(len(documents))

500


In [6]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [7]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [8]:
df.shape

(500, 12)

In [9]:
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="../chroma_db"
)

In [10]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

In [11]:
question = "Which titles in the context are about crime?"

relevant_docs = retriever.invoke(question)

for doc in relevant_docs:
    print(doc.page_content)
    print("-" * 80)

Title: Day and Night. Category: TV Show. Genre: Crime TV Shows, International TV Shows, TV Dramas. Director: Unknown. Cast: Pan Yueming, Wang Longzheng, Liang Yuen, Lü Xiaolin, Yin Shuyi, Hou Xuelong, Song Naigang. Country: China. Release date: March 23, 2018. Rating: TV-MA. Duration: 1 Season. Description: A detective assists with an investigation into a brutal mass murder case in which his twin brother is the prime suspect.
--------------------------------------------------------------------------------
Title: Dark Crimes. Category: Movie. Genre: Dramas, Thrillers. Director: Alexandros Avranas. Cast: Jim Carrey, Marton Csokas, Charlotte Gainsbourg, Agata Kulesza, Kati Outinen, Zbigniew Zamachowski, Danuta Kowalska, Vlad Ivanov, Robert Więckiewicz, Piotr Głowacki. Country: United Kingdom, Poland, United States. Release date: October 15, 2019. Rating: R. Duration: 93 min. Description: A detective on a cold murder case discovers that a famous writer’s latest novel contains details chill

In [12]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [13]:
context = "\n\n".join([doc.page_content for doc in relevant_docs])

prompt = f"""
You are answering questions about a Netflix dataset.
All titles in the context come from the Netflix dataset.

Use only the context below to answer the question.
If a title seems related to the question, mention the title and explain why.

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print(response.content)

All titles in the context are about crime:

*   **Day and Night** is about crime because its genre is "Crime TV Shows" and its description mentions "a brutal mass murder case" and a "detective" investigating.
*   **Dark Crimes** is about crime because its description states it's about "A detective on a cold murder case" and "the crime he’s investigating."
*   **Gomorrah** is about crime because its genre is "Crime TV Shows" and its description focuses on "Mafia activity."
*   **Wanted** is about crime because its genre is "Crime TV Shows" and its description involves characters who "witness a murder involving dirty cops and are framed for the crime."
*   **El Marginal** is about crime because its genre is "Crime TV Shows" and its description mentions "investigating a kidnapping" and "dangerous felons" in a prison.


In [14]:
def ask_rag(question):
    
    relevant_docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in relevant_docs]
    )

    prompt = f"""
    You are answering questions about a Netflix dataset.

    All titles in the context come from the Netflix dataset.

    Use ONLY the context below.

    If multiple titles match the question:
    - list ALL matching titles
    - include release year if available

    Answer clearly using bullet points.

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    return response.content

In [15]:
print(ask_rag(
    "How many movies are produced in Canada?"
))

*   There are 5 movies produced in Canada:
    *   Adventures in Public School (2018)
    *   Invisible Essence: The Little Prince (2019)
    *   Rampage: President Down (2017)
    *   The Art of the Steal (2018)
    *   The Body Remembers When the World Broke Open (2019)


In [16]:
print(ask_rag(
    "How many movies are produced since 2015?"
))

*   Theater of Life (2017)
*   Bomb Scared (2017)
*   50/50 (2019)
*   Rumble (2017)
*   My Travel Buddy (2019)

There are 5 movies produced since 2015.


In [17]:
print(ask_rag(
    "Horror movie 2010+?"
))

*   Train of the Dead (2018)
*   #Alive (2020)
*   Silent Hill: Revelation (2019)
*   Cam (2018)
*   Hungerford (2018)


In [ ]:
print(ask_rag(
    "Description of kaal and rattlesnake?"
))

*   **Kaal (2020):** Trapped in a national park, a tiger expert and a group of hunters turn to an enigmatic local for help when they mysteriously begin to die one by one.
*   **Rattlesnake:** Not found in the context.


In [19]:
print(ask_rag(
    "List all the movies/shows that contains the genre drama"
))

Here are the movies that contain the genre drama:

*   Layla M. (2018)
*   Surat Dari Praha (2018)
*   The Reconquest (2017)
*   Posesif (2020)
*   The Show (2020)


In [23]:
print(ask_rag(
    "List 5 comedy movies released 2020"
))

Here are 5 comedy movies released in 2020:

*   Either Me Or My Auntie (2020)
*   Ginny Weds Sunny (2020)
*   Karkar (2020)
*   A Chaster Marriage (2020)
*   Instructions Not Included (2020)


In [22]:
print(ask_rag(
    "List 5 comedy movies released 2020 and show me the description of them"
))

Here are 5 comedy movies released in 2020:

*   **Either Me Or My Auntie** (2020): A musician's marriage proposal to his girlfriend is denied by her mother, whose affinity for magic begins to meddle in their relationship even more.
*   **Instructions Not Included** (2020): Unable to locate the elusive mother of a baby girl left on his doorstep, an Acapulco playboy unexpectedly begins to develop feelings for the tot.
*   **A Chaster Marriage** (2020): Forced to wed his childhood friend, a man obsessed with football attempts to get rid of his wife in order to keep the woman he truly loves.
*   **Karkar** (2020): After electrocution leaves a wealthy man brain-damaged, his money-hungry aunt and uncle try to con him by marrying him off to a fake relative.
*   **Ginny Weds Sunny** (2020): Eager to marry but constantly rejected by women, a bachelor hopes to win over a former crush by accepting help from an unlikely source: her mother.
